In [2]:
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

In [ ]:
# Instancia del modelo local
llm = ChatOllama(
    model="tinyllama:1.1b-chat",
    base_url="http://localhost:11434",
    temperature=0.7,
)

# Estructura de estado (solo llevaremos el mensaje del usuario)
class AgentState(dict):
    msg: str

# Nodo que llama al modelo y muestra la respuesta
def chat_node(state: AgentState) -> AgentState:
    reply = llm.invoke(state["msg"])
    print("\n🤖", reply.content, "\n")
    return state  # sin modificar el estado por ahora

# Construcción del grafo
g = StateGraph(AgentState)
g.add_node("CHAT", chat_node)

# Puntos CLAVE: entrypoint y salida
g.add_edge(START, "CHAT")   # entrada
g.add_edge("CHAT", END)     # salida

bot = g.compile()           # compila el grafo

if __name__ == "__main__":
    while True:
        user = input("Tú > ")
        if user.lower() in {"salir", "exit"}:
            break
        bot.invoke({"msg": user})
